# Create Irish Research Council Awards

Creates Irish Research Council (IRC) awards from the council's own public
**Search Awardees Database** (https://research.ie/awardees/). The page loads
its table from `https://research.ie/awardees_search/search_new.php`, a plain
server-rendered endpoint queried one award year at a time. One row per
awardee: Year, Name, Award Category, Award (programme), Higher Education
Institution, Partner, Discipline, Project Title. **11,759 awardee rows 1999-2026 in the raw table; 6,745 IRC-era (2012-2024) rows shipped, ~98% with a project title, 100% awardee name + institution** (local run 2026-09-30).

No bulk file exists for the IRC-era database (runbook ladder item 0 checked
2026-09-30): data.gov.ie carries only Research Ireland's post-merger "Grant
Commitments" CSV (Aug 2024 onward, funder = Research Ireland) and the old SFI
file.

**Which rows ship under the IRC funder ID (the IRC -> Research Ireland line):**
the database mixes three eras, all shown on research.ie.
- **1999-2011**: awards of the IRC's predecessors, IRCSET and IRCHSS, which
  have their own OpenAlex funders (F4320320842, F4320320843). Not shipped here.
- **2012-2024**: the IRC era (IRC formed 2012 from IRCSET + IRCHSS; merged with
  SFI into Research Ireland on 1 Aug 2024; the 2024 award year is the IRC's own
  2024 calls). **Shipped.**
- **2025+**: Research Ireland calls (Research Ireland, F8766430056). Not shipped.
The raw table keeps every year; the filter is `year BETWEEN 2012 AND 2024` below.

**Prerequisites:**
- Run `scripts/local/irish_research_council_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/irish_research_council/irish_research_council_projects.parquet`
  (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** the database publishes **no award number**.
Citing works mostly quote IRC application references such as `GOIPG/2019/2511`,
`EPSPG/2016/85`, `IRCLA/2017/285`, `COALESCE/2020/13` (citation-side sample,
crossref_work.grants for F4320321056, 2026-09-30), which are not published for
IRC-era awards anywhere public, so these rows cannot collapse onto those
~4,300 stubs. The key is synthetic and deterministic:
`IRC-{year}-{award-slug}-{name-slug}` (e.g.
`IRC-2019-postgraduate-scholarship-programme-aaron-barron`), with `-2`, `-3`
for the rare same-person/award/year repeats; the script raises on collision.

**No amounts** are published (scholarship/fellowship values are standard per
scheme but not per award): Step 6.7 amount check waived. **Dates:** only the
award year is published; `start_date` = 1 January of the award year (runbook
§2.3 year pattern), `end_date` NULL.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320321056`
- display_name: Irish Research Council
- ror_id / doi: from `openalex.funders.funders` (ror 051xex213, doi 10.13039/501100002081)

**Priority:** `490` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.irish_research_council_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/irish_research_council/irish_research_council_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_rows,
       SUM(CASE WHEN TRY_CAST(year AS INT) BETWEEN 2012 AND 2024 THEN 1 ELSE 0 END) as irc_era_rows
FROM openalex.awards.irish_research_council_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.irish_research_council_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.irish_research_council_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320321056 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320321056;

## Step 2: Create Irish Research Council Awards Table (IRC era 2012-2024 only)

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.irish_research_council_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320321056  -- Irish Research Council
),

irc_era AS (
    SELECT * FROM openalex.awards.irish_research_council_raw
    WHERE TRY_CAST(year AS INT) BETWEEN 2012 AND 2024
      AND funder_award_id IS NOT NULL AND TRIM(funder_award_id) != ''
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE LOWER(TRIM(g.award_category))
        WHEN 'early career' THEN 'fellowship'
        WHEN 'principal investigator-led' THEN 'research'
        ELSE 'grant'
    END as funding_type,
    NULLIF(TRIM(g.award), '') as funder_scheme,
    'irc_awardees_database' as provenance,
    TRY_TO_DATE(CONCAT(g.year, '-01-01'), 'yyyy-MM-dd') as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.year AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    named_struct(
        'given_name', NULLIF(TRIM(g.given_name), ''),
        'family_name', NULLIF(TRIM(g.family_name), ''),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.institution), ''),
            'country', 'IE',
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>)
        as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>)
        as investigators,
    'https://research.ie/awardees/' as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM irc_era g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Shape check (§2.1.1): synthetic IRC-{year}-... key, IRC-era years only, unique.
SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT id) AS distinct_ids,
    SUM(CASE WHEN funder_award_id RLIKE '^IRC-20(1[2-9]|2[0-4])-[a-z0-9-]+$' THEN 0 ELSE 1 END) AS bad_shape,
    MIN(start_year) AS min_year, MAX(start_year) AS max_year
FROM openalex.awards.irish_research_council_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'irc_awardees_database' AND priority = 490;

-- Insert into openalex_awards_raw with priority.
-- Priority 490: direct-from-funder ingest of the IRC awardees database (IRC era 2012-2024).
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    490 as priority
FROM openalex.awards.irish_research_council_awards;

## Verification Queries

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name
FROM openalex.awards.irish_research_council_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) as cnt
FROM openalex.awards.irish_research_council_awards
GROUP BY 1, 2 ORDER BY cnt DESC LIMIT 60;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.irish_research_council_awards
GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.irish_research_council_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.irish_research_council_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 coverage (amount waived: not published).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution,
    COUNT(amount) as has_amount
FROM openalex.awards.irish_research_council_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'irc_awardees_database'
GROUP BY provenance, priority;